# Análise Exploratória — Previsão Financeira Diária

**Mazine IA & DATA**

Exploração da base histórica sintética de receita e despesa diária por unidade de negócio, preparando o terreno para o modelo XGBoost treinado em `main.py`.

---
**Seções:**
1. Carregamento e visão geral
2. Série temporal consolidada
3. Análise por unidade
4. Sazonalidade (dia da semana, mês)
5. Distribuição e outliers
6. Correlação e saldo acumulado
7. Decomposição de tendência
8. Previsão vs Real (backtesting visual)
9. Conclusões para o modelo

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates

plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.color': '#e0e0e0',
    'grid.linewidth': 0.7,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})

COR_RECEITA = '#2ecc71'
COR_DESPESA = '#e74c3c'
COR_SALDO   = '#3498db'
COR_PREV    = '#f39c12'

def brl(v):
    return f'R$ {v:,.0f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

print('Bibliotecas carregadas.')

## 1. Carregamento e visão geral

In [ ]:
df_u = pd.read_csv('dados/base_diaria_unidades.csv', parse_dates=['ds'])
df   = pd.read_csv('dados/base_diaria.csv',          parse_dates=['ds'])

df_u['saldo'] = df_u['receita'] - df_u['despesa']
df['saldo']   = df['receita']   - df['despesa']

unidades = sorted(df_u['unidade'].unique())

print(f'Base por unidade : {len(df_u):,} linhas | {len(unidades)} unidades')
print(f'Período          : {df["ds"].min().date()} → {df["ds"].max().date()}')
print(f'Dias no histórico: {df["ds"].nunique():,}')
df_u.head(10)

In [ ]:
print('=== Estatísticas Gerais (consolidado) ===')
desc = df[['receita', 'despesa', 'saldo']].describe().T
for col in ['mean', 'min', '50%', 'max']:
    desc[col] = desc[col].map(brl)
display(desc[['count', 'mean', 'min', '50%', 'max', 'std']])

dias_zero = (df['receita'] == 0).sum()
print(f'\nDias com receita zero: {dias_zero} ({dias_zero/len(df)*100:.1f}%) — domingos/feriados')

## 2. Série temporal consolidada

In [ ]:
df_m = df.set_index('ds').resample('ME')[['receita', 'despesa', 'saldo']].sum().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.bar(df_m['ds'], df_m['receita'], width=20, label='Receita', color=COR_RECEITA, alpha=0.85)
ax.bar(df_m['ds'], df_m['despesa'], width=20, label='Despesa', color=COR_DESPESA, alpha=0.85)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax.set_title('Receita e Despesa Mensal Consolidada')
ax.legend()

ax2 = axes[1]
cores = [COR_RECEITA if v >= 0 else COR_DESPESA for v in df_m['saldo']]
ax2.bar(df_m['ds'], df_m['saldo'], width=20, color=cores, alpha=0.85)
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax2.set_title('Saldo Mensal (Receita − Despesa)')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
df_plot = df[df['receita'] > 0].copy()
df_plot['mm30'] = df_plot['receita'].rolling(30).mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_plot['ds'], df_plot['receita'], color=COR_RECEITA, alpha=0.3, linewidth=0.8, label='Receita diária')
ax.plot(df_plot['ds'], df_plot['mm30'],    color=COR_RECEITA, linewidth=2.0, label='Média móvel 30d')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax.set_title('Receita Diária Consolidada — série completa')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Análise por unidade

In [ ]:
resumo = (
    df_u[df_u['receita'] > 0]
    .groupby('unidade')[['receita', 'despesa', 'saldo']]
    .mean()
    .round(0)
)
resumo.columns = ['Receita Média/dia', 'Despesa Média/dia', 'Saldo Médio/dia']
for col in resumo.columns:
    resumo[col] = resumo[col].map(brl)
display(resumo)

In [ ]:
rec_u = df_u.groupby('unidade')['receita'].sum().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(rec_u.index, rec_u.values, color=COR_RECEITA, alpha=0.85)
for bar, val in zip(bars, rec_u.values):
    ax.text(val * 1.01, bar.get_y() + bar.get_height()/2, brl(val), va='center', fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax.set_title('Receita Total por Unidade')
plt.tight_layout()
plt.show()

In [ ]:
df_u_m = (
    df_u.set_index('ds')
    .groupby('unidade')
    .resample('ME')['receita'].sum()
    .reset_index()
)

cols_grid = 5
rows_grid = (len(unidades) + cols_grid - 1) // cols_grid
fig, axes = plt.subplots(rows_grid, cols_grid, figsize=(18, rows_grid * 3), sharey=False)
axes = axes.flatten()

for i, u in enumerate(unidades):
    dado = df_u_m[df_u_m['unidade'] == u]
    axes[i].bar(dado['ds'], dado['receita'], width=20, color=COR_RECEITA, alpha=0.8)
    axes[i].set_title(f'Unidade {u}')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Receita Mensal por Unidade', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Sazonalidade

In [ ]:
df_util = df[df['receita'] > 0].copy()
df_util['dia_semana'] = df_util['ds'].dt.dayofweek
df_util['mes']        = df_util['ds'].dt.month

DIAS_PT  = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb']
MESES_PT = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sem = df_util.groupby('dia_semana')[['receita', 'despesa']].mean()
x = sem.index
axes[0].bar(x - 0.2, sem['receita'], width=0.4, label='Receita', color=COR_RECEITA, alpha=0.85)
axes[0].bar(x + 0.2, sem['despesa'], width=0.4, label='Despesa', color=COR_DESPESA, alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels([DIAS_PT[i] for i in x])
axes[0].set_title('Média por Dia da Semana')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: brl(v)))
axes[0].legend()

mes = df_util.groupby('mes')[['receita', 'despesa']].mean()
x2 = mes.index
axes[1].bar(x2 - 0.2, mes['receita'], width=0.4, label='Receita', color=COR_RECEITA, alpha=0.85)
axes[1].bar(x2 + 0.2, mes['despesa'], width=0.4, label='Despesa', color=COR_DESPESA, alpha=0.85)
axes[1].set_xticks(x2)
axes[1].set_xticklabels([MESES_PT[m - 1] for m in x2], rotation=45)
axes[1].set_title('Média por Mês')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: brl(v)))
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
pivot = df_util.pivot_table(values='receita', index='dia_semana', columns='mes', aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGn')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([DIAS_PT[i] for i in pivot.index])
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([MESES_PT[m - 1] for m in pivot.columns])
ax.set_title('Receita Média — Dia da Semana × Mês')
plt.colorbar(im, ax=ax, label='Receita Média (R$)')

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val/1000:.0f}k', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 5. Distribuição e outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col, cor, label in [
    (axes[0], 'receita', COR_RECEITA, 'Receita'),
    (axes[1], 'despesa', COR_DESPESA, 'Despesa'),
]:
    dados = df_util[col]
    ax.hist(dados, bins=60, color=cor, alpha=0.75, edgecolor='white')
    p1, p99 = dados.quantile(0.01), dados.quantile(0.99)
    ax.axvline(p1,           color='black',  linestyle='--', linewidth=1.2, label=f'P1: {brl(p1)}')
    ax.axvline(p99,          color='navy',   linestyle='--', linewidth=1.2, label=f'P99: {brl(p99)}')
    ax.axvline(dados.mean(), color='orange', linestyle='-',  linewidth=1.5, label=f'Média: {brl(dados.mean())}')
    ax.set_title(f'Distribuição — {label}')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dados_rec  = [df_u[df_u['unidade'] == u]['receita'].values for u in unidades]
dados_desp = [df_u[df_u['unidade'] == u]['despesa'].values for u in unidades]

for ax, dados, cor, titulo in [
    (axes[0], dados_rec,  COR_RECEITA, 'Receita por Unidade'),
    (axes[1], dados_desp, COR_DESPESA, 'Despesa por Unidade'),
]:
    bp = ax.boxplot(dados, labels=unidades, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor(cor)
        patch.set_alpha(0.6)
    ax.set_title(titulo)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Correlação e saldo acumulado

In [ ]:
corr_val = df_util[['receita', 'despesa']].corr().loc['receita', 'despesa']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(df_util['receita'], df_util['despesa'], alpha=0.25, s=8, color=COR_SALDO)
axes[0].set_xlabel('Receita')
axes[0].set_ylabel('Despesa')
axes[0].set_title(f'Receita × Despesa  (r = {corr_val:.2f})')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
axes[0].tick_params(axis='x', rotation=30)

df_sal = df.set_index('ds').resample('ME')['saldo'].sum().cumsum().reset_index()
axes[1].fill_between(df_sal['ds'], df_sal['saldo'], alpha=0.3, color=COR_SALDO)
axes[1].plot(df_sal['ds'], df_sal['saldo'], color=COR_SALDO, linewidth=2)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
axes[1].set_title('Saldo Acumulado')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Decomposição de tendência (média móvel)

In [ ]:
serie = df_util.set_index('ds')['receita']
tendencia    = serie.rolling(30, center=True).mean()
sazonalidade = serie - tendencia
residuo      = sazonalidade - sazonalidade.rolling(7, center=True).mean()

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(serie.index,     serie.values,     color=COR_RECEITA, alpha=0.35, linewidth=0.8)
axes[0].plot(tendencia.index, tendencia.values, color='darkgreen', linewidth=2)
axes[0].set_title('Série Original + Tendência (MM30)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))

axes[1].plot(sazonalidade.index, sazonalidade.values, color=COR_SALDO, alpha=0.6, linewidth=0.8)
axes[1].axhline(0, color='black', linewidth=0.6, linestyle='--')
axes[1].set_title('Componente Sazonal')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))

axes[2].plot(residuo.index, residuo.values, color='gray', alpha=0.5, linewidth=0.7)
axes[2].axhline(0, color='black', linewidth=0.6, linestyle='--')
axes[2].set_title('Resíduo')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8. Previsão vs Real (backtesting visual)

Simulamos um backtesting: treinamos o modelo com os primeiros **80% do histórico** e comparamos a previsão com os **20% finais** que o modelo nunca viu.

Isso prova visualmente que o modelo aprendeu os padrões reais da série — e não apenas memorizou os dados de treino.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from treinadores.treinar_xgboost import (
    add_calendar_features, add_target_lags, clip_outliers, CALENDAR_COLS
)

LAGS = [1, 7, 14, 30]
ROLLS = [7, 30]
GW = [7, 30]
ALVO = 'receita'

# Usa base consolidada (sem unidade) para o backtesting visual
df_bt = df[df['receita'] > 0].sort_values('ds').reset_index(drop=True).copy()
df_bt[ALVO], _ = clip_outliers(df_bt[ALVO])

df_bt = add_calendar_features(df_bt)
df_bt = add_target_lags(df_bt, ALVO, LAGS, ROLLS, GW)
df_bt = df_bt.dropna().reset_index(drop=True)

features = (
    CALENDAR_COLS
    + [f'{ALVO}_lag{l}' for l in LAGS]
    + [f'{ALVO}_roll{r}' for r in ROLLS]
    + [f'{ALVO}_crescimento_{g}d' for g in GW]
    + [f'{ALVO}_tend_{g}d' for g in GW]
)

corte = int(len(df_bt) * 0.80)
treino = df_bt.iloc[:corte]
teste  = df_bt.iloc[corte:]

modelo = XGBRegressor(
    n_estimators=800, learning_rate=0.05, max_depth=6,
    subsample=0.9, colsample_bytree=0.9,
    objective='reg:squarederror', random_state=42
)
modelo.fit(treino[features], treino[ALVO])
previsoes_bt = modelo.predict(teste[features])

mae  = mean_absolute_error(teste[ALVO], previsoes_bt)
mape = mean_absolute_percentage_error(teste[ALVO], previsoes_bt) * 100

print(f'Período de teste : {teste["ds"].min().date()} → {teste["ds"].max().date()}')
print(f'Dias avaliados   : {len(teste)}')
print(f'MAE              : {brl(mae)}/dia')
print(f'MAPE             : {mape:.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# --- Gráfico 1: série completa com destaque no período de teste ---
ax = axes[0]
ax.plot(treino['ds'], treino[ALVO], color=COR_RECEITA, linewidth=1.0,
        alpha=0.6, label='Histórico (treino)')
ax.plot(teste['ds'], teste[ALVO], color=COR_RECEITA, linewidth=1.5,
        label='Real (teste)')
ax.plot(teste['ds'], previsoes_bt, color=COR_PREV, linewidth=2.0,
        linestyle='--', label='Previsto pelo modelo')
ax.axvline(teste['ds'].iloc[0], color='gray', linestyle=':', linewidth=1.2,
           label='Início do período de teste')
ax.set_title('Receita — Real vs Previsto (backtesting 20% final)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
ax.tick_params(axis='x', rotation=45)

# --- Gráfico 2: zoom no período de teste com erro preenchido ---
ax2 = axes[1]
ax2.plot(teste['ds'], teste[ALVO], color=COR_RECEITA, linewidth=1.5, label='Real')
ax2.plot(teste['ds'], previsoes_bt, color=COR_PREV, linewidth=2.0,
         linestyle='--', label='Previsto')
ax2.fill_between(teste['ds'], teste[ALVO], previsoes_bt,
                 alpha=0.15, color='gray', label='Erro')
ax2.set_title(f'Zoom — Período de Teste  |  MAE: {brl(mae)}/dia  |  MAPE: {mape:.1f}%')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d/%b/%Y'))
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Scatter real vs previsto — quanto mais próximo da diagonal, melhor
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(teste[ALVO], previsoes_bt, alpha=0.4, s=15, color=COR_PREV)
lim_min = min(teste[ALVO].min(), previsoes_bt.min()) * 0.95
lim_max = max(teste[ALVO].max(), previsoes_bt.max()) * 1.05
ax.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', linewidth=1.2, label='Previsão perfeita')
ax.set_xlabel('Real')
ax.set_ylabel('Previsto')
ax.set_title('Real vs Previsto — dispersão')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: brl(x)))
ax.tick_params(axis='both', rotation=30)
ax.legend()
plt.tight_layout()
plt.show()

## 9. Conclusões para o modelo

| Observação | Impacto no modelo |
|---|---|
| Domingos com receita zero | `dia_semana` é feature essencial |
| Pico de receita na sexta-feira | Sazonalidade semanal capturada por `lag_7` e `dia_semana` |
| Pico em dezembro/janeiro | `mes` e `dia_ano` como features de calendário |
| Concentração nos primeiros dias do mês | `janela_inicio_mes_5d` e `dia` capturam esse padrão |
| Tendência de crescimento suave | `roll_mean_30` e features de crescimento modelam isso |
| Alta correlação receita × despesa | Modelos separados por alvo são adequados |
| Outliers presentes mas contidos | Clipping por quantil (P0.5 / P99.5) aplicado no treino |

**Mínimo recomendado para treino:** 6 meses por unidade para estabilizar lags de 30 e 365 dias.

---
*Notebook desenvolvido por Mazine IA & DATA — dados 100% sintéticos para fins de portfólio.*